# 02 — Análisis exploratorio (EDA)

Trabajamos con `data/clean/bcsc_clean.parquet`. Recuerda: cada fila es una **combinación de factores** y la columna `count` indica cuántas mamografías reales corresponden a esa combinación. Toda agregación debe ponderarse por `count`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

df = pd.read_parquet(Path('../data/clean/bcsc_clean.parquet'))
print(df.shape)
df.head()

In [ ]:
# Distribución del target ponderada
tgt = df.groupby('breast_cancer_history')['count'].sum()
print(tgt)
print(f'\nPrevalencia de cáncer: {tgt[1] / tgt.sum():.4%}')
tgt.plot.bar(title='Mamografías por estado de diagnóstico'); plt.show()

In [ ]:
# Tasa de cáncer por grupo de edad
def tasa_por(col):
    g = df.groupby(col).apply(
        lambda x: (x.loc[x['breast_cancer_history'] == 1, 'count'].sum() / x['count'].sum())
    )
    return g

tasa_por('age_group_5_years').plot.bar(title='Tasa de cáncer por grupo de edad'); plt.show()
tasa_por('first_degree_hx').plot.bar(title='Tasa por antecedentes familiares'); plt.show()
tasa_por('BIRADS_breast_density').plot.bar(title='Tasa por densidad mamaria'); plt.show()
tasa_por('bmi_group').plot.bar(title='Tasa por BMI'); plt.show()

In [ ]:
# Heatmap de correlación (sobre filas expandidas con sample para no estallar memoria)
sample = df.sample(min(50000, len(df)), weights='count', random_state=42)
feats = [c for c in sample.columns if c not in ('count', 'year')]
corr = sample[feats].replace(9, np.nan).corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlación entre factores'); plt.show()

## Hallazgos esperados

- Edad y densidad BI-RADS son los predictores más fuertes.
- Antecedentes familiares 1er grado: aumenta la tasa ~2x.
- Prevalencia base baja (~1%): el modelo necesitará atención al desbalance (class_weight).